# PyTorch Tensors

This notebook adapts the official PyTorch tensor quickstart tutorial for this Quarto Book.

Source: [PyTorch Tutorials - Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)

Learning goals:

- Create tensors from Python data, NumPy arrays, and other tensors.
- Inspect tensor shape, dtype, and device.
- Move tensors to available accelerators when appropriate.
- Use indexing, concatenation, matrix multiplication, element-wise operations, and in-place operations.
- Understand how CPU tensors and NumPy arrays can share memory.

Tensors are specialized data structures similar to arrays and matrices. In PyTorch, tensors encode model inputs, outputs, and parameters.

Compared with NumPy arrays, tensors can run on GPUs or other accelerators, can share memory with NumPy arrays on CPU, and are integrated with automatic differentiation. We will return to gradients in the [Autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) tutorial.


In [1]:
import numpy as np
import torch

_ = torch.manual_seed(42)

## Initializing a Tensor

Tensors can be initialized in several ways.

### Directly from data

Tensors can be created directly from data. The data type is automatically inferred.

In [2]:
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

### From a NumPy array

Tensors can be created from NumPy arrays and NumPy arrays can be created from tensors. We will inspect the memory-sharing behavior near the end of this notebook.

In [3]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

### From another tensor

The new tensor retains the shape and dtype of the source tensor unless explicitly overridden.

In [4]:
x_ones = torch.ones_like(x_data)  # Retains the properties of x_data
print(f"Ones Tensor: \n{x_ones}\n")

x_rand = torch.rand_like(x_data, dtype=torch.float)  # Overrides the dtype of x_data
print(f"Random Tensor: \n{x_rand}\n")

Ones Tensor: 
tensor([[1, 1],
        [1, 1]])

Random Tensor: 
tensor([[0.8823, 0.9150],
        [0.3829, 0.9593]])



### With random or constant values

`shape` is a tuple of tensor dimensions. In the functions below, it determines the dimensionality of the output tensor.

In [5]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n{rand_tensor}\n")
print(f"Ones Tensor: \n{ones_tensor}\n")
print(f"Zeros Tensor: \n{zeros_tensor}")

Random Tensor: 
tensor([[0.3904, 0.6009, 0.2566],
        [0.7936, 0.9408, 0.1332]])

Ones Tensor: 
tensor([[1., 1., 1.],
        [1., 1., 1.]])

Zeros Tensor: 
tensor([[0., 0., 0.],
        [0., 0., 0.]])


> What to notice: tensor creation APIs often preserve metadata from an existing tensor unless you override it. This is useful when you want a new tensor with the same shape or dtype as a reference tensor.

## Attributes of a Tensor

Tensor attributes describe shape, dtype, and the device where tensor storage lives.

In [6]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


## Operations on Tensors

PyTorch includes tensor operations for arithmetic, linear algebra, matrix manipulation, indexing, slicing, sampling, and more. The full API is described in the [`torch` reference](https://pytorch.org/docs/stable/torch.html).

Most tensor operations can run on CPU or on an available accelerator such as CUDA, MPS, MTIA, or XPU. By default, tensors are created on CPU. Move tensors explicitly with `.to(...)` after checking accelerator availability. Copying large tensors across devices can be expensive.

In [7]:
# Move the tensor to the current accelerator if one is available.
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

Try out some of the operations from the list. If you\'re familiar with
the NumPy API, you\'ll find the Tensor API a breeze to use.


### Standard NumPy-like indexing and slicing

In [8]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:, 1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


### Joining tensors

Use `torch.cat` to concatenate tensors along an existing dimension. See also [`torch.stack`](https://pytorch.org/docs/stable/generated/torch.stack.html), which creates a new dimension while joining tensors.

In [9]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


### Arithmetic operations

In [10]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

### Single-element tensors

A one-element tensor can be converted to a Python numerical value with `.item()`.

In [11]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

12.0 <class 'float'>


### In-place operations

Operations that store results into the original operand are called in-place operations. They are usually marked by a trailing `_`, for example `x.copy_(y)` or `x.t_()`.

In [12]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


> Note: in-place operations can save memory, but they can also interfere with autograd because they immediately overwrite values needed for gradient computation. Prefer non-in-place operations unless memory pressure or an API contract makes mutation useful.

## Bridge with NumPy {#bridge-to-np-label}

CPU tensors and NumPy arrays can share their underlying memory. Changing one can change the other.

### Tensor to NumPy array

In [13]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


A change in the tensor reflects in the NumPy array.


In [14]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


### NumPy array to Tensor

In [15]:
n = np.ones(5)
t = torch.from_numpy(n)

Changes in the NumPy array are reflected in the tensor.

In [16]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


> What to notice: this memory sharing applies to CPU tensors. Once tensors live on an accelerator, moving between Tensor and NumPy requires an explicit transfer back to CPU.